# Remote Tesseract OCR server — for Google Colab

Serves Tesseract over FastAPI + a Cloudflare tunnel, exactly like the PaddleOCR
server notebook. Run the cells top to bottom; the last cells print a **url** and
a **token**.

**The API is deliberately identical to the PaddleOCR server**, so the local
layout notebook works against it with no code change — paste this url and token
into its settings cell and run cells 12→14 as usual.

| | |
|---|---|
| `GET /health` | `Authorization: Bearer <token>` |
| `POST /infer` | form `page` (int ≥ 1), file `file`, same auth |

The response reuses PaddleOCR's field names (`rec_texts`, `rec_scores`,
`rec_polys`), so the local `iter_rec_blocks` extractor finds the text unchanged:

```json
{"ok": true, "page": 4, "api": "tesseract",
 "raw": [{"res": {"rec_texts": ["..."], "rec_scores": [0.93], "rec_polys": [...]}}]}
```

> Run this on Colab, not locally — it depends on `apt-get`, `/content`, and
> `cloudflared` for linux.

In [ ]:
# ============================================================
# 1. Install Tesseract + the Arabic language data
# ============================================================
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara tesseract-ocr-osd

!python -m pip install -q \
  pytesseract \
  fastapi \
  uvicorn \
  python-multipart \
  requests \
  opencv-python-headless

In [ ]:
# ============================================================
# 2. Verify the install
# ============================================================
import shutil
import subprocess

import cv2
import pytesseract

print("tesseract binary :", shutil.which("tesseract"))
print(subprocess.run(["tesseract", "--version"], capture_output=True,
                     text=True).stdout.splitlines()[0])
print("pytesseract      :", pytesseract.get_tesseract_version())
print("opencv           :", cv2.__version__)
print()
print("languages        :", pytesseract.get_languages(config=""))

assert "ara" in pytesseract.get_languages(config=""), \
    "Arabic language data missing -- re-run cell 1"
print("\nArabic language data present.")

In [ ]:
# ============================================================
# 3. OPTIONAL: better Arabic accuracy (tessdata_best)
# ============================================================
# Ubuntu ships the "fast" Arabic model. tessdata_best is noticeably more
# accurate on scanned Arabic, at maybe 2-3x the CPU time per image.
# Skip this cell to keep the default model.

USE_TESSDATA_BEST = True

import glob
import os
import shutil

if USE_TESSDATA_BEST:
    tessdata_dirs = glob.glob("/usr/share/tesseract-ocr/*/tessdata")
    if not tessdata_dirs:
        raise RuntimeError("tessdata directory not found -- re-run cell 1")

    tessdata = tessdata_dirs[0]
    target = os.path.join(tessdata, "ara.traineddata")
    print("tessdata dir:", tessdata)

    if not os.path.exists(target + ".fast_backup"):
        shutil.copy2(target, target + ".fast_backup")
        print("backed up the shipped model -> ara.traineddata.fast_backup")

    url = ("https://github.com/tesseract-ocr/tessdata_best/raw/main/"
           "ara.traineddata")
    !wget -q -O /tmp/ara_best.traineddata {url}

    size_mb = os.path.getsize("/tmp/ara_best.traineddata") / 1024**2
    if size_mb < 1:
        raise RuntimeError("download failed -- /tmp/ara_best.traineddata is tiny")

    shutil.move("/tmp/ara_best.traineddata", target)
    print(f"installed tessdata_best ara.traineddata ({size_mb:.1f} MB)")
else:
    print("Skipped -- using the Ubuntu default Arabic model.")

In [ ]:
%%writefile /content/tesseract_server.py

import os
import tempfile
import threading
from pathlib import Path
from typing import Optional

import cv2
import numpy as np
import pytesseract
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from pytesseract import Output

API_TOKEN = os.getenv("OCR_API_TOKEN", "").strip()
MAX_UPLOAD_BYTES = 30 * 1024 * 1024

# Defaults, overridable per request via the /infer form fields.
# psm 6 = "assume a single uniform block of text", which is what a layout crop
# is. psm 3 (fully automatic) does its own page segmentation and tends to be
# worse on an already-cropped block.
DEFAULT_LANG = os.getenv("TESS_LANG", "ara")
DEFAULT_PSM = os.getenv("TESS_PSM", "6")
DEFAULT_OEM = os.getenv("TESS_OEM", "3")

# Tesseract wants ~300 dpi text. Crops rendered at 200 dpi read better upscaled.
DEFAULT_SCALE = float(os.getenv("TESS_SCALE", "1.5"))
DEFAULT_BINARIZE = os.getenv("TESS_BINARIZE", "1") == "1"

# Tesseract is not thread safe across a shared process; serialize like the
# PaddleOCR server does.
tess_lock = threading.Lock()


# ============================================================
# Image preparation
# ============================================================

def prepare_image(path, scale, binarize):
    """Grey -> optional upscale -> optional Otsu. Returns (image, scale_used)."""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"could not decode image: {path}")

    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if scale and scale != 1.0:
        grey = cv2.resize(grey, None, fx=scale, fy=scale,
                          interpolation=cv2.INTER_CUBIC)
    else:
        scale = 1.0

    if binarize:
        # Otsu on a grey scan; Tesseract binarizes internally anyway, but doing
        # it here with a global threshold is more predictable on clean scans.
        _, grey = cv2.threshold(grey, 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return grey, scale


# ============================================================
# Word rows -> lines
# ============================================================

def lines_from_data(data, scale):
    """Group Tesseract's word rows into lines.

    image_to_data returns one row per word plus structural rows carrying
    conf == -1. Words are grouped by (block, paragraph, line); coordinates are
    divided back by `scale` so they refer to the original crop's pixels.
    """
    groups = {}
    order = []

    for i in range(len(data["text"])):
        text = (data["text"][i] or "").strip()
        try:
            conf = float(data["conf"][i])
        except (TypeError, ValueError):
            continue

        # conf < 0 marks a structural row, not a recognized word.
        if not text or conf < 0:
            continue

        key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
        if key not in groups:
            groups[key] = {"words": [], "confs": [], "boxes": []}
            order.append(key)

        left, top = data["left"][i], data["top"][i]
        width, height = data["width"][i], data["height"][i]

        groups[key]["words"].append(text)
        groups[key]["confs"].append(conf)
        groups[key]["boxes"].append((left, top, left + width, top + height))

    texts, scores, polys, boxes = [], [], [], []

    for key in order:
        group = groups[key]

        # Tesseract emits words in reading order, including right-to-left for
        # Arabic, so joining in sequence keeps the logical order.
        texts.append(" ".join(group["words"]))
        scores.append(round(sum(group["confs"]) / len(group["confs"]) / 100.0, 4))

        x1 = min(b[0] for b in group["boxes"]) / scale
        y1 = min(b[1] for b in group["boxes"]) / scale
        x2 = max(b[2] for b in group["boxes"]) / scale
        y2 = max(b[3] for b in group["boxes"]) / scale

        x1, y1, x2, y2 = (round(v, 2) for v in (x1, y1, x2, y2))
        boxes.append([x1, y1, x2, y2])
        polys.append([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])

    return texts, scores, polys, boxes


# ============================================================
# Auth
# ============================================================

def require_auth(authorization: Optional[str]):
    if not API_TOKEN:
        raise HTTPException(status_code=500,
                            detail="OCR_API_TOKEN is not configured")

    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")


# ============================================================
# FastAPI
# ============================================================

app = FastAPI(title="Colab Tesseract Inference Server", version="1.0.0")


@app.get("/health")
def health(authorization: Optional[str] = Header(default=None)):
    require_auth(authorization)

    return {
        "status": "ok",
        "engine": "tesseract",
        "tesseract_version": str(pytesseract.get_tesseract_version()),
        "pytesseract_version": getattr(pytesseract, "__version__", None),
        "languages": pytesseract.get_languages(config=""),

        # Mirrors the PaddleOCR server's keys so the same client code can read
        # either server's /health without special-casing.
        "paddle_version": None,
        "paddleocr_version": None,
        "device": "cpu",

        "config": {
            "language": DEFAULT_LANG,
            "psm": DEFAULT_PSM,
            "oem": DEFAULT_OEM,
            "scale": DEFAULT_SCALE,
            "binarize": DEFAULT_BINARIZE,
        },
    }


@app.post("/infer")
async def infer(
    page: int = Form(...),
    file: UploadFile = File(...),
    lang: Optional[str] = Form(default=None),
    psm: Optional[str] = Form(default=None),
    oem: Optional[str] = Form(default=None),
    scale: Optional[float] = Form(default=None),
    binarize: Optional[bool] = Form(default=None),
    authorization: Optional[str] = Header(default=None),
):
    require_auth(authorization)

    if page < 1:
        raise HTTPException(status_code=422, detail="page must be >= 1")

    content = await file.read()
    if not content:
        raise HTTPException(status_code=400, detail="Empty file")
    if len(content) > MAX_UPLOAD_BYTES:
        raise HTTPException(status_code=413, detail="Image too large")

    suffix = Path(file.filename or "crop.png").suffix.lower()
    if suffix not in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff",
                      ".webp"}:
        suffix = ".png"

    use_lang = lang or DEFAULT_LANG
    use_psm = psm or DEFAULT_PSM
    use_oem = oem or DEFAULT_OEM
    use_scale = DEFAULT_SCALE if scale is None else scale
    use_binarize = DEFAULT_BINARIZE if binarize is None else binarize

    config = f"--oem {use_oem} --psm {use_psm}"
    tmp_path = None

    try:
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp.write(content)
            tmp_path = tmp.name

        image, used_scale = prepare_image(tmp_path, use_scale, use_binarize)

        with tess_lock:
            data = pytesseract.image_to_data(
                image, lang=use_lang, config=config, output_type=Output.DICT
            )

        texts, scores, polys, boxes = lines_from_data(data, used_scale)

        return {
            "ok": True,
            "page": page,
            "api": "tesseract",
            "raw": [{
                "res": {
                    "input_path": file.filename,
                    "page_index": None,

                    # PaddleOCR field names on purpose -- lets the same client
                    # parse either server's response.
                    "rec_texts": texts,
                    "rec_scores": scores,
                    "rec_polys": polys,
                    "rec_boxes": boxes,

                    "engine": "tesseract",
                    "full_text": "\n".join(texts),
                    "tesseract": {
                        "lang": use_lang,
                        "psm": use_psm,
                        "oem": use_oem,
                        "scale": used_scale,
                        "binarize": use_binarize,
                    },
                }
            }],
        }

    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(status_code=500, detail=repr(exc))

    finally:
        if tmp_path:
            try:
                os.remove(tmp_path)
            except OSError:
                pass

In [ ]:
# ============================================================
# 5. Generate the API token
# ============================================================
import os
import secrets

TOKEN = secrets.token_urlsafe(32)
os.environ["OCR_API_TOKEN"] = TOKEN

# Server-side OCR defaults. Change these, then re-run the uvicorn cell.
os.environ["TESS_LANG"] = "ara"
os.environ["TESS_PSM"] = "6"
os.environ["TESS_OEM"] = "3"
os.environ["TESS_SCALE"] = "1.5"
os.environ["TESS_BINARIZE"] = "1"

print("OCR API TOKEN:")
print(TOKEN)

In [ ]:
# ============================================================
# 6. Start the server
# ============================================================
import subprocess
import sys
import time

import requests

SERVER_LOG = "/content/tesseract_server.log"

# Free port 8000 if an earlier run left uvicorn behind.
print("Terminating any existing uvicorn process ...")
try:
    pids = subprocess.check_output(
        ["pgrep", "-f", "uvicorn tesseract_server:app"]
    ).decode().strip().split("\n")
    for pid in pids:
        if pid:
            subprocess.run(["kill", "-9", pid], check=False)
            print(f"  killed pid {pid}")
    time.sleep(1)
except subprocess.CalledProcessError:
    print("  none found")

log_file = open(SERVER_LOG, "w")

server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "tesseract_server:app",
     "--host", "127.0.0.1", "--port", "8000", "--workers", "1"],
    cwd="/content",
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

headers = {"Authorization": f"Bearer {TOKEN}"}

for _ in range(120):
    try:
        response = requests.get("http://127.0.0.1:8000/health",
                                headers=headers, timeout=2)
        if response.status_code == 200:
            print("\nServer up:")
            for key, value in response.json().items():
                print(f"  {key}: {value}")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    log_file.close()
    print(open(SERVER_LOG).read())
    server_process.terminate()
    server_process.wait(timeout=5)
    raise RuntimeError("Tesseract server did not start correctly")

In [ ]:
# ============================================================
# 7. Local smoke test  --  before exposing it publicly
# ============================================================
import cv2
import numpy as np
import requests

# A synthetic image with Latin text: proves the request path, preprocessing and
# line grouping all work. Arabic glyphs need a font Colab may not have, so the
# real Arabic check is cell 11 with one of your own crops.
canvas = np.full((160, 700), 255, dtype=np.uint8)
cv2.putText(canvas, "Tesseract 12345", (20, 70), cv2.FONT_HERSHEY_SIMPLEX,
            1.4, 0, 3, cv2.LINE_AA)
cv2.putText(canvas, "second line", (20, 130), cv2.FONT_HERSHEY_SIMPLEX,
            1.1, 0, 2, cv2.LINE_AA)
cv2.imwrite("/content/smoke.png", canvas)

with open("/content/smoke.png", "rb") as fh:
    response = requests.post(
        "http://127.0.0.1:8000/infer",
        headers={"Authorization": f"Bearer {TOKEN}"},
        data={"page": 1, "lang": "eng"},
        files={"file": ("smoke.png", fh, "image/png")},
        timeout=120,
    )

print("status:", response.status_code)
payload = response.json()
result = payload["raw"][0]["res"]

print("lines :", result["rec_texts"])
print("scores:", result["rec_scores"])
print("boxes :", result["rec_boxes"])

assert response.status_code == 200 and result["rec_texts"], \
    "smoke test produced no text -- check the server log"
print("\nRequest path, preprocessing and line grouping all work.")

In [ ]:
# ============================================================
# 8. Install cloudflared
# ============================================================
!curl -L \
  --output /tmp/cloudflared.deb \
  "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"

!dpkg -i /tmp/cloudflared.deb

In [ ]:
# ============================================================
# 9. Open the tunnel
# ============================================================
import re
import subprocess
import time

!pkill -f cloudflared || true
time.sleep(1)

TUNNEL_LOG = "/content/cloudflared.log"

with open(TUNNEL_LOG, "w") as f:
    tunnel_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=f,
        stderr=subprocess.STDOUT,
    )

OCR_BASE_URL = None

for _ in range(60):
    try:
        text = open(TUNNEL_LOG, encoding="utf-8", errors="ignore").read()
    except Exception:
        text = ""

    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if match:
        OCR_BASE_URL = match.group(0)
        break

    time.sleep(1)

if OCR_BASE_URL is None:
    print(open(TUNNEL_LOG).read())
    raise RuntimeError("Could not obtain a Cloudflare URL")

print("=" * 70)
print("PASTE THESE INTO THE LOCAL NOTEBOOK'S SETTINGS CELL")
print("=" * 70)
print(f'OCR_BASE_URL = "{OCR_BASE_URL}"')
print(f'OCR_TOKEN    = "{TOKEN}"')
print("=" * 70)
print("\nBoth values change whenever this notebook restarts.")
print("Keep this Colab tab open -- the tunnel dies with the runtime.")

In [ ]:
# ============================================================
# 10. Verify through the public url
# ============================================================
import time

import requests

time.sleep(5)  # give the tunnel a moment to propagate

response = requests.get(
    OCR_BASE_URL + "/health",
    headers={"Authorization": f"Bearer {TOKEN}"},
    timeout=30,
)

print("status:", response.status_code)
for key, value in response.json().items():
    print(f"  {key}: {value}")

assert response.status_code == 200, "public url is not reachable"
print("\nReachable from outside. The local notebook can now use it.")

In [ ]:
# ============================================================
# 11. OPTIONAL: test with one of your real Arabic crops
# ============================================================
# Upload a single crop PNG from your local ./crops/ folder to check Arabic
# accuracy end to end before running all 134 through it.

from google.colab import files as colab_files
import requests

uploaded = colab_files.upload()
crop_name = next(iter(uploaded.keys()))

with open(crop_name, "rb") as fh:
    response = requests.post(
        OCR_BASE_URL + "/infer",
        headers={"Authorization": f"Bearer {TOKEN}"},
        data={"page": 1},
        files={"file": (crop_name, fh, "image/png")},
        timeout=300,
    )

print("status:", response.status_code)
result = response.json()["raw"][0]["res"]

print("settings:", result["tesseract"])
print(f"\n{len(result['rec_texts'])} line(s):\n")
for text, score in zip(result["rec_texts"], result["rec_scores"]):
    print(f"  {score:.3f}  {text}")

## Tuning

Change these in cell 5, then re-run cell 6 (the tunnel in cell 9 keeps running,
and both url and token stay valid):

| env var | default | notes |
|---|---|---|
| `TESS_PSM` | `6` | `6` = uniform text block, right for a layout crop. Try `4` for columns, `7` for a single line, `3` for full auto page segmentation. |
| `TESS_SCALE` | `1.5` | Upscale before OCR. Tesseract wants ~300 dpi text; crops rendered at 200 dpi read better at 1.5–2.0. |
| `TESS_BINARIZE` | `1` | Otsu before OCR. Set `0` to pass the grey image through untouched. |
| `TESS_LANG` | `ara` | `ara+eng` for mixed pages. |
| `TESS_OEM` | `3` | `3` = default LSTM. `1` forces LSTM only. |

`/infer` also accepts `lang`, `psm`, `oem`, `scale` and `binarize` as form
fields, so you can override any of them per request without restarting.

## Expect Tesseract to be slower and less accurate than PP-OCRv5 here

On scanned Arabic, PP-OCRv5 with `arabic_PP-OCRv5_mobile_rec` generally beats
Tesseract, and `tessdata_best` costs more CPU than the fast model. Both servers
speak the same API, so run the same 134 crops through each and compare
`mean_rec_score` in `ocr_boxes.csv` — but note the two engines compute
confidence differently, so treat the score gap as a hint and read the text side
by side before concluding.

To keep both result sets, change `OUTPUT_DIR` in the local notebook before the
second run — otherwise `ocr_raw/` is reused and `OCR_RESUME` will serve the
cached PaddleOCR answers instead of calling Tesseract.